
# Joint-model kinetic analysis with actual-time correction

This notebook starts from the **wide normalized-volume CSV** that has not yet been ratio-corrected.

It performs:

1. parsing of the wide CSV into a long-format table,
2. use of the actual measurement time columns,
3. optional QC/filtering using the absolute time-bin deviation columns,
4. direct ratio correction against mock for comparison,
5. primary fitting with a global joint model:

\[
B(t)=P_B + (1-P_B)e^{-k_Bt}
\]

\[
C_j(t)=B(t)\times\left[P_j+(1-P_j)e^{-k_jt}\right]
\]

where `mock` estimates the baseline \(B(t)\), and each antibody condition has its own antibody-specific decay component.

## Time handling

The default is:

```python
TIME_MODE = "elapsed_actual"
```

This uses:

\[
t_\mathrm{fit} =
\texttt{mean_actual_time_min} -
\texttt{first_mean_actual_time_min}
\]

for each condition.

The column `*_mean_abs_time_bin_deviation_min` is retained as a QC value and can optionally be used to exclude poorly centered time bins, but it is **not** treated as a y-axis fitting weight.


In [ ]:
from pathlib import Path
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares

In [ ]:

# -----------------------------
# User settings
# -----------------------------

INPUT_CSV = Path(r" *** \combined_normalized_volume_wide_only2p5h.csv")

OUTPUT_DIR = Path(r" *** \combined_normalized_volume\kinetics_outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# The keys are display names used in output tables/plots.
# The values are the prefixes used in the wide CSV column names.
CONDITION_PREFIXES = {
    "mock": "mock",
    "Nivo": "Nivo",
    "Durva": "Durva",
    "Enva": "Enva",
    "mw11h317": "mw11h317",
    "aPD1": "aPD",
    "aPDL1": "aPDL",
}

CONTROL_LABEL = "mock"

ANTIBODY_LABELS = [
    "Nivo",
    "Durva",
    "Enva",
    "mw11h317",
    "aPD1",
    "aPDL1",
]

# -----------------------------
# Additional shared-plateau / fixed-plateau comparison settings
# -----------------------------
# These extra models do not replace the independent-plateau joint model above.
#
# 1) SHARED_PLATEAU_MODEL_LABELS:
#    A strict comparison model where all listed antibodies share one plateau P_shared
#    but each antibody keeps its own k.
#    aPDL1 is excluded by default because it trends upward/non-decay in the current data;
#    add it here if you explicitly want to test the poor-fit constrained model.
SHARED_PLATEAU_MODEL_LABELS = [
    "Nivo",
    "Durva",
    "Enva",
    "mw11h317",
    "aPD1",
]

# 2) SHARED_PLATEAU_REFERENCE_LABELS:
#    Conditions used to estimate a low/completed-clearing plateau.
#    Based on the microscopy observation, Durva and Enva are used by default.
SHARED_PLATEAU_REFERENCE_LABELS = [
    "Durva",
    "Enva",
]

# 3) FIXED_PLATEAU_APPLY_LABELS:
#    Conditions refitted with the Durva/Enva-inferred plateau fixed.
#    This is a sensitivity analysis asking:
#    "If this condition ultimately reached the Durva/Enva plateau, what k would explain the data?"
FIXED_PLATEAU_APPLY_LABELS = [
    "Nivo",
    "Durva",
    "Enva",
    "mw11h317",
    "aPD1",
    "aPDL1",
]

# Time modes:
# "elapsed_actual" = actual time minus each condition's first actual time. Recommended here.
# "actual"         = use absolute mean_actual_time_min directly.
# "nominal"        = use the binned time_min column.
TIME_MODE = "elapsed_actual"

# Fitting weights:
# "sd"   = weight by SD column. This matches the earlier SD-weighted Origin-style workflow.
# "sem"  = weight by SEM column. This is often better if fitting mean curves.
# "none" = unweighted fit.
ERROR_FOR_FIT = "sd"

# The normalized t=0 point has zero SD/SEM because it is the normalization anchor.
# It is excluded from weighted fitting by default.
EXCLUDE_T0_FROM_FIT = True

# Optional QC filter using *_mean_abs_time_bin_deviation_min.
# Example: set to 2.0 to exclude bins whose mean absolute deviation from nominal bin center exceeds 2 min.
MAX_ALLOWED_BIN_DEVIATION_MIN = None

# Boundaries for the joint model.
K_BASE_MAX = 0.2
P_BASE_MAX = 1.5
K_EFFECT_MAX = 0.5
P_EFFECT_MAX = 2.0

# Values of k below this threshold are treated as effectively zero/non-identifiable
# for fit-status labels and half-time reporting in constrained sensitivity models.
MIN_INTERPRETABLE_K = 1e-5


In [ ]:

# -----------------------------
# Data loading and parsing
# -----------------------------

def find_prefixed_column(df, prefix, suffix, required=True):
    """
    Find an exact prefixed column like '{prefix}_{suffix}'.
    Matching is case-insensitive as a fallback.
    """
    expected = f"{prefix}_{suffix}"
    if expected in df.columns:
        return expected

    lower_map = {str(c).lower(): c for c in df.columns}
    if expected.lower() in lower_map:
        return lower_map[expected.lower()]

    if required:
        raise KeyError(f"Missing expected column: {expected}")
    return None


def load_wide_normalized_csv(input_csv, condition_prefixes, time_mode="elapsed_actual"):
    """
    Convert the wide normalized-volume CSV into a long table with one row per condition/time point.
    """
    wide = pd.read_csv(input_csv)
    wide = wide.loc[:, ~wide.columns.astype(str).str.startswith("Unnamed")].copy()

    records = []

    for label, prefix in condition_prefixes.items():
        time_col = find_prefixed_column(wide, prefix, "time_min", required=False)
        actual_col = find_prefixed_column(wide, prefix, "mean_actual_time_min", required=False)
        dev_col = find_prefixed_column(wide, prefix, "mean_abs_time_bin_deviation_min", required=False)
        mean_col = find_prefixed_column(wide, prefix, "mean_normalized_volume", required=True)
        sd_col = find_prefixed_column(wide, prefix, "sd_normalized_volume", required=False)
        sem_col = find_prefixed_column(wide, prefix, "sem_normalized_volume", required=False)
        n_col = find_prefixed_column(wide, prefix, "total_n", required=False)
        n_files_col = find_prefixed_column(wide, prefix, "n_source_files", required=False)
        n_rows_col = find_prefixed_column(wide, prefix, "n_source_rows", required=False)

        tmp = pd.DataFrame({
            "condition": label,
            "source_prefix": prefix,
            "time_min": pd.to_numeric(wide[time_col], errors="coerce") if time_col else np.nan,
            "mean_actual_time_min": pd.to_numeric(wide[actual_col], errors="coerce") if actual_col else np.nan,
            "mean_abs_time_bin_deviation_min": pd.to_numeric(wide[dev_col], errors="coerce") if dev_col else np.nan,
            "mean_normalized_volume": pd.to_numeric(wide[mean_col], errors="coerce"),
            "sd_normalized_volume": pd.to_numeric(wide[sd_col], errors="coerce") if sd_col else np.nan,
            "sem_normalized_volume": pd.to_numeric(wide[sem_col], errors="coerce") if sem_col else np.nan,
            "total_n": pd.to_numeric(wide[n_col], errors="coerce") if n_col else np.nan,
            "n_source_files": pd.to_numeric(wide[n_files_col], errors="coerce") if n_files_col else np.nan,
            "n_source_rows": pd.to_numeric(wide[n_rows_col], errors="coerce") if n_rows_col else np.nan,
        }).dropna(subset=["mean_normalized_volume"]).copy()

        if time_mode == "actual":
            tmp["fit_time_min"] = tmp["mean_actual_time_min"]
        elif time_mode == "nominal":
            tmp["fit_time_min"] = tmp["time_min"]
        elif time_mode == "elapsed_actual":
            actual = tmp["mean_actual_time_min"]
            if actual.notna().any():
                tmp["fit_time_min"] = actual - actual.dropna().iloc[0]
            else:
                nominal = tmp["time_min"]
                tmp["fit_time_min"] = nominal - nominal.dropna().iloc[0]
        else:
            raise ValueError("time_mode must be 'elapsed_actual', 'actual', or 'nominal'")

        records.append(tmp)

    long = pd.concat(records, ignore_index=True)
    long = long.sort_values(["condition", "fit_time_min"]).reset_index(drop=True)
    return long


def get_sigma(df, error_for_fit="sd"):
    """
    Return the y-axis uncertainty used for fitting.
    Falls back to the other uncertainty column if the requested one is unavailable.
    """
    if error_for_fit == "none":
        return pd.Series(np.ones(len(df)), index=df.index, dtype=float)

    if error_for_fit not in {"sd", "sem"}:
        raise ValueError("error_for_fit must be 'sd', 'sem', or 'none'")

    primary = "sd_normalized_volume" if error_for_fit == "sd" else "sem_normalized_volume"
    fallback = "sem_normalized_volume" if primary == "sd_normalized_volume" else "sd_normalized_volume"

    sigma = df[primary].astype(float).copy()
    if fallback in df.columns:
        sigma = sigma.where(np.isfinite(sigma) & (sigma > 0), df[fallback].astype(float))

    return sigma


long = load_wide_normalized_csv(INPUT_CSV, CONDITION_PREFIXES, time_mode=TIME_MODE)

print(f"Loaded: {INPUT_CSV}")
print(f"Rows: {len(long)}")
display(
    long.groupby("condition")
    .agg(
        n_points=("mean_normalized_volume", "size"),
        first_nominal_time=("time_min", "first"),
        first_actual_time=("mean_actual_time_min", "first"),
        last_fit_time=("fit_time_min", "last"),
        final_normalized_volume=("mean_normalized_volume", "last"),
        max_abs_bin_deviation=("mean_abs_time_bin_deviation_min", "max"),
    )
)


In [ ]:

# -----------------------------
# Joint model
# -----------------------------

def baseline_model(t, k_base, p_base):
    """
    Baseline/mock drift:
    B(t) = P_base + (1 - P_base) * exp(-k_base * t)
    """
    t = np.asarray(t, dtype=float)
    return p_base + (1.0 - p_base) * np.exp(-k_base * t)


def effect_model(t, k, p):
    """
    Antibody-specific effect:
    E(t) = P + (1 - P) * exp(-k * t)
    """
    t = np.asarray(t, dtype=float)
    return p + (1.0 - p) * np.exp(-k * t)


def make_fit_table(
    long,
    labels,
    error_for_fit="sd",
    exclude_t0=True,
    max_bin_dev=None,
):
    """
    Prepare the table used in fitting and flag included rows.
    """
    d = long[long["condition"].isin(labels)].copy()
    d["sigma"] = get_sigma(d, error_for_fit)

    mask = np.isfinite(d["fit_time_min"]) & np.isfinite(d["mean_normalized_volume"])

    if error_for_fit != "none":
        mask &= np.isfinite(d["sigma"]) & (d["sigma"] > 0)
    else:
        d["sigma"] = 1.0

    if exclude_t0:
        mask &= d["fit_time_min"] > 0

    if max_bin_dev is not None:
        mask &= (
            d["mean_abs_time_bin_deviation_min"].isna()
            | (d["mean_abs_time_bin_deviation_min"] <= max_bin_dev)
        )

    d["included_in_fit"] = mask
    return d


def fit_global_joint_model(
    long,
    control_label,
    antibody_labels,
    error_for_fit="sd",
    exclude_t0=True,
    max_bin_dev=None,
):
    """
    Fit all curves simultaneously:

    mock:
        y = B(t)

    antibody j:
        y = B(t) * E_j(t)
    """
    labels = [control_label] + list(antibody_labels)
    fit_df = make_fit_table(
        long,
        labels,
        error_for_fit=error_for_fit,
        exclude_t0=exclude_t0,
        max_bin_dev=max_bin_dev,
    )
    included = fit_df[fit_df["included_in_fit"]].copy()

    if included.empty:
        raise ValueError("No valid data points are included in the fit.")

    # Initial baseline guesses from mock
    mock_all = long[long["condition"] == control_label].sort_values("fit_time_min")
    p_base0 = float(np.nanmedian(mock_all["mean_normalized_volume"].tail(5)))
    p_base0 = float(np.clip(p_base0, 0.0, min(P_BASE_MAX, 1.2)))
    k_base0 = 0.005

    x0 = [k_base0, p_base0]
    lb = [0.0, 0.0]
    ub = [K_BASE_MAX, P_BASE_MAX]

    antibody_labels = list(antibody_labels)

    for ab in antibody_labels:
        ab_all = long[long["condition"] == ab].sort_values("fit_time_min")

        if not ab_all.empty:
            t_med = float(np.nanmedian(ab_all["fit_time_min"].tail(5)))
            b_med = baseline_model(t_med, k_base0, p_base0)
            p0 = float(np.nanmedian(ab_all["mean_normalized_volume"].tail(5)) / b_med) if b_med != 0 else 0.5
        else:
            p0 = 0.5

        p0 = float(np.clip(p0, 0.0, P_EFFECT_MAX))

        x0 += [0.03, p0]
        lb += [0.0, 0.0]
        ub += [K_EFFECT_MAX, P_EFFECT_MAX]

    def unpack(params):
        k_base, p_base = params[0], params[1]
        effects = {}
        idx = 2
        for ab in antibody_labels:
            effects[ab] = (params[idx], params[idx + 1])
            idx += 2
        return k_base, p_base, effects

    def predict_rows(params, rows):
        k_base, p_base, effects = unpack(params)
        t = rows["fit_time_min"].to_numpy(dtype=float)
        b = baseline_model(t, k_base, p_base)
        pred = np.empty(len(rows), dtype=float)

        conds = rows["condition"].to_numpy()
        for i, cond in enumerate(conds):
            if cond == control_label:
                pred[i] = b[i]
            else:
                k, p = effects[cond]
                pred[i] = b[i] * effect_model(t[i], k, p)

        return pred

    def residuals(params):
        pred = predict_rows(params, included)
        y = included["mean_normalized_volume"].to_numpy(dtype=float)
        sigma = included["sigma"].to_numpy(dtype=float)
        return (pred - y) / sigma

    res = least_squares(
        residuals,
        x0,
        bounds=(lb, ub),
        max_nfev=100000,
        loss="linear",
    )

    # Approximate covariance from the Jacobian.
    m = len(res.fun)
    n_params = len(res.x)
    dof = max(1, m - n_params)
    red_chi2 = float(np.sum(res.fun ** 2) / dof)

    try:
        jtj_inv = np.linalg.pinv(res.jac.T @ res.jac)
        cov = jtj_inv * red_chi2
        se = np.sqrt(np.diag(cov))
    except Exception:
        cov = None
        se = np.full_like(res.x, np.nan, dtype=float)

    fit_all = fit_df.copy()
    fit_all["model_y"] = predict_rows(res.x, fit_all)

    k_base, p_base, effects = unpack(res.x)
    fit_all["baseline_y"] = baseline_model(fit_all["fit_time_min"], k_base, p_base)
    fit_all["effect_corrected_by_baseline_fit"] = (
        fit_all["mean_normalized_volume"] / fit_all["baseline_y"]
    )
    fit_all.loc[
        fit_all["condition"] == control_label,
        "effect_corrected_by_baseline_fit"
    ] = np.nan

    return res, cov, se, fit_all, unpack, red_chi2


res, cov, se, fit_all, unpack, red_chi2 = fit_global_joint_model(
    long,
    control_label=CONTROL_LABEL,
    antibody_labels=ANTIBODY_LABELS,
    error_for_fit=ERROR_FOR_FIT,
    exclude_t0=EXCLUDE_T0_FROM_FIT,
    max_bin_dev=MAX_ALLOWED_BIN_DEVIATION_MIN,
)

print("Fit success:", res.success)
print("Fit message:", res.message)
print("Reduced chi-square-like value:", red_chi2)


In [ ]:

# -----------------------------
# Metrics and direct ratio correction
# -----------------------------

CI_MULTIPLIER = 1.96  # normal-approximation 95% CI from nonlinear least-squares covariance


def symmetric_ci_from_se(estimate, se, lower_bound=None, upper_bound=None, multiplier=CI_MULTIPLIER):
    """
    Return low/high confidence limits from estimate ± multiplier*SE.
    Optional clipping keeps bounded parameters inside their allowed range.
    """
    if not (np.isfinite(estimate) and np.isfinite(se)):
        return np.nan, np.nan

    low = estimate - multiplier * se
    high = estimate + multiplier * se

    if lower_bound is not None:
        low = max(lower_bound, low)
    if upper_bound is not None:
        high = min(upper_bound, high)

    return low, high


def half_life_from_k(k):
    return np.log(2) / k if np.isfinite(k) and k > 0 else np.nan


def half_life_ci_from_k_ci(k_low, k_high):
    """
    Transform a k CI into a t1/2 CI.

    Since t1/2 = ln(2)/k, the interval reverses:
        t1/2_low  = ln(2)/k_high
        t1/2_high = ln(2)/k_low

    If k_low is clipped to zero, the upper t1/2 CI is undefined/infinite.
    """
    if not (np.isfinite(k_low) and np.isfinite(k_high)) or k_high <= 0:
        return np.nan, np.nan

    t_half_low = np.log(2) / k_high
    t_half_high = np.inf if k_low <= 0 else np.log(2) / k_low
    return t_half_low, t_half_high


def t_half_se_delta_method(k, k_se):
    """
    Approximate SE for t1/2 using the delta method.
    This is useful to report, but the CI from transformed k limits is preferred.
    """
    if np.isfinite(k) and k > 0 and np.isfinite(k_se):
        return (np.log(2) / (k ** 2)) * k_se
    return np.nan


def add_rate_uncertainty_fields(row, k, k_se, p=None, p_se=None, has_interpretable_half_time=True):
    """
    Add k, k SE/CI, t1/2, t1/2 SE/CI, and optional plateau uncertainty to one metrics row.
    """
    k_ci_low, k_ci_high = symmetric_ci_from_se(k, k_se, lower_bound=0.0)
    t_half = half_life_from_k(k) if has_interpretable_half_time else np.nan
    t_half_se = t_half_se_delta_method(k, k_se) if has_interpretable_half_time else np.nan

    if has_interpretable_half_time:
        t_half_ci_low, t_half_ci_high = half_life_ci_from_k_ci(k_ci_low, k_ci_high)
    else:
        t_half_ci_low, t_half_ci_high = np.nan, np.nan

    row.update({
        "k_min_inv": k,
        "k_se_min_inv": k_se,
        "k_ci95_low_min_inv": k_ci_low,
        "k_ci95_high_min_inv": k_ci_high,
        "t_half_min": t_half,
        "t_half_se_min": t_half_se,
        "t_half_ci95_low_min": t_half_ci_low,
        "t_half_ci95_high_min": t_half_ci_high,
        "tau_min": 1 / k if np.isfinite(k) and k > 0 and has_interpretable_half_time else np.nan,
    })

    if p is not None:
        p_ci_low, p_ci_high = symmetric_ci_from_se(p, p_se, lower_bound=0.0, upper_bound=P_EFFECT_MAX)
        row.update({
            "plateau_P": p,
            "P_se": p_se,
            "P_ci95_low": p_ci_low,
            "P_ci95_high": p_ci_high,
        })

    return row


def compute_metrics(long, fit_all, res, se, unpack, control_label, antibody_labels):
    k_base, p_base, effects = unpack(res.x)

    param_names = ["k_base", "P_base"]
    for ab in antibody_labels:
        param_names += [f"{ab}_k", f"{ab}_P"]
    se_map = dict(zip(param_names, se))

    rows = []

    # Baseline control
    mock_obs = long[long["condition"] == control_label].sort_values("fit_time_min")

    base_row = {
        "condition": control_label,
        "role": "baseline_control",
        "fit_status": "baseline fitted",
        "clearable_fraction_1_minus_P": np.nan,
        "t90_min": np.nan,
        "t95_min": np.nan,
        "observed_y_final": float(mock_obs["mean_normalized_volume"].iloc[-1]),
        "observed_min_y": float(mock_obs["mean_normalized_volume"].min()),
        "observed_effect_corrected_final": np.nan,
        "observed_effect_corrected_min": np.nan,
        "n_fit_points": int(fit_all[(fit_all["condition"] == control_label) & fit_all["included_in_fit"]].shape[0]),
        "r2_raw_normalized": np.nan,
        "rmse_raw_normalized": np.nan,
    }

    base_row = add_rate_uncertainty_fields(
        base_row,
        k=k_base,
        k_se=se_map.get("k_base", np.nan),
        p=p_base,
        p_se=se_map.get("P_base", np.nan),
        has_interpretable_half_time=True,
    )
    rows.append(base_row)

    for ab in antibody_labels:
        k, p = effects[ab]
        k_se = se_map.get(f"{ab}_k", np.nan)
        p_se = se_map.get(f"{ab}_P", np.nan)

        has_decay = (np.isfinite(k) and k > MIN_INTERPRETABLE_K) and (p < 1)

        t90 = np.log(10) / k if has_decay else np.nan
        t95 = np.log(20) / k if has_decay else np.nan

        sub = fit_all[fit_all["condition"] == ab].sort_values("fit_time_min").copy()
        y = sub["mean_normalized_volume"].to_numpy(dtype=float)
        pred = sub["model_y"].to_numpy(dtype=float)

        # Report raw-model fit quality excluding the t=0 normalization anchor
        mask = sub["fit_time_min"].to_numpy(dtype=float) > 0
        if mask.sum() > 1:
            ss_res = np.sum((y[mask] - pred[mask]) ** 2)
            ss_tot = np.sum((y[mask] - np.mean(y[mask])) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            rmse = np.sqrt(np.mean((y[mask] - pred[mask]) ** 2))
        else:
            r2 = np.nan
            rmse = np.nan

        fit_status = "decay_fit_ok" if has_decay else "no_decay_or_increase__do_not_interpret_half_time"

        row = {
            "condition": ab,
            "role": "antibody",
            "fit_status": fit_status,
            "clearable_fraction_1_minus_P": 1 - p if has_decay else np.nan,
            "t90_min": t90,
            "t95_min": t95,
            "observed_y_final": float(sub["mean_normalized_volume"].iloc[-1]),
            "observed_min_y": float(sub["mean_normalized_volume"].min()),
            "observed_effect_corrected_final": float(sub["effect_corrected_by_baseline_fit"].iloc[-1]),
            "observed_effect_corrected_min": float(sub["effect_corrected_by_baseline_fit"].min()),
            "n_fit_points": int(sub[sub["included_in_fit"]].shape[0]),
            "r2_raw_normalized": r2,
            "rmse_raw_normalized": rmse,
        }

        row = add_rate_uncertainty_fields(
            row,
            k=k,
            k_se=k_se,
            p=p,
            p_se=p_se,
            has_interpretable_half_time=has_decay,
        )

        rows.append(row)

    return pd.DataFrame(rows)


def make_direct_ratio_correction_table(long, control_label, antibody_labels):
    """
    Direct measured-mock ratio correction:
        R(t) = C_norm(t) / M_norm(t)

    Because actual elapsed times can differ slightly between conditions, mock is linearly interpolated
    onto each antibody condition's elapsed actual time grid.

    SD and SEM are propagated using first-order ratio error propagation.
    """
    mock = long[long["condition"] == control_label].sort_values("fit_time_min").copy()

    t_mock = mock["fit_time_min"].to_numpy(dtype=float)
    y_mock = mock["mean_normalized_volume"].to_numpy(dtype=float)
    sd_mock = mock["sd_normalized_volume"].to_numpy(dtype=float)
    sem_mock = mock["sem_normalized_volume"].to_numpy(dtype=float)

    unique_t, unique_idx = np.unique(t_mock, return_index=True)
    t_mock = t_mock[unique_idx]
    y_mock = y_mock[unique_idx]
    sd_mock = sd_mock[unique_idx]
    sem_mock = sem_mock[unique_idx]

    rows = []

    for ab in antibody_labels:
        cond = long[long["condition"] == ab].sort_values("fit_time_min").copy()

        t = cond["fit_time_min"].to_numpy(dtype=float)
        c = cond["mean_normalized_volume"].to_numpy(dtype=float)
        sd_c = cond["sd_normalized_volume"].to_numpy(dtype=float)
        sem_c = cond["sem_normalized_volume"].to_numpy(dtype=float)

        m_interp = np.interp(t, t_mock, y_mock)
        sd_m_interp = np.interp(t, t_mock, sd_mock)
        sem_m_interp = np.interp(t, t_mock, sem_mock)

        ratio = c / m_interp

        with np.errstate(divide="ignore", invalid="ignore"):
            ratio_sd = ratio * np.sqrt((sd_c / c) ** 2 + (sd_m_interp / m_interp) ** 2)
            ratio_sem = ratio * np.sqrt((sem_c / c) ** 2 + (sem_m_interp / m_interp) ** 2)

        # t=0 is the normalization anchor
        zero = np.isclose(t, 0)
        ratio[zero] = 1.0
        ratio_sd[zero] = 0.0
        ratio_sem[zero] = 0.0

        tmp = cond[[
            "condition",
            "source_prefix",
            "time_min",
            "mean_actual_time_min",
            "fit_time_min",
            "mean_abs_time_bin_deviation_min",
            "mean_normalized_volume",
            "sd_normalized_volume",
            "sem_normalized_volume",
            "total_n",
        ]].copy()

        tmp = tmp.rename(columns={
            "condition": "antibody_condition",
            "mean_normalized_volume": "condition_mean_normalized_volume",
            "sd_normalized_volume": "condition_sd_normalized_volume",
            "sem_normalized_volume": "condition_sem_normalized_volume",
        })

        tmp["mock_interpolated_mean_normalized_volume"] = m_interp
        tmp["mock_interpolated_sd_normalized_volume"] = sd_m_interp
        tmp["mock_interpolated_sem_normalized_volume"] = sem_m_interp
        tmp["direct_ratio_corrected_mean"] = ratio
        tmp["direct_ratio_corrected_sd_prop"] = ratio_sd
        tmp["direct_ratio_corrected_sem_prop"] = ratio_sem

        rows.append(tmp)

    return pd.concat(rows, ignore_index=True)



metrics = compute_metrics(
    long=long,
    fit_all=fit_all,
    res=res,
    se=se,
    unpack=unpack,
    control_label=CONTROL_LABEL,
    antibody_labels=ANTIBODY_LABELS,
)

direct_ratio = make_direct_ratio_correction_table(
    long=long,
    control_label=CONTROL_LABEL,
    antibody_labels=ANTIBODY_LABELS,
)

display_cols = [
    "condition",
    "fit_status",
    "k_min_inv",
    "k_se_min_inv",
    "k_ci95_low_min_inv",
    "k_ci95_high_min_inv",
    "t_half_min",
    "t_half_ci95_low_min",
    "t_half_ci95_high_min",
    "plateau_P",
    "clearable_fraction_1_minus_P",
    "t90_min",
    "t95_min",
    "r2_raw_normalized",
    "observed_effect_corrected_final",
]

display(metrics[[c for c in display_cols if c in metrics.columns]].round(4))


In [ ]:

# -----------------------------
# Save output tables and plots
# -----------------------------


def compute_fit_quality_summary(
    fit_points,
    model_name,
    condition_col="condition",
    observed_col="mean_normalized_volume",
    predicted_col="model_y",
    sigma_col="sigma",
    included_col="included_in_fit",
    time_col="fit_time_min",
):
    """
    Compute fit-quality diagnostics from a fit-point table.

    The main R²/RMSE values are calculated on the rows used for fitting
    (included_in_fit == True). This avoids the normalized t=0 anchor when
    EXCLUDE_T0_FROM_FIT=True.

    Weighted residual diagnostics use sigma if it is present and nonzero.
    Because the joint models share parameters across conditions, the
    per-condition reduced chi-square values are approximate diagnostics rather
    than formal hypothesis tests.
    """
    rows = []

    def _one_row(sub, label):
        valid = (
            sub[observed_col].notna()
            & sub[predicted_col].notna()
        )
        included = valid.copy()
        if included_col in sub.columns:
            included = included & sub[included_col].astype(bool)

        y = sub.loc[included, observed_col].to_numpy(dtype=float)
        pred = sub.loc[included, predicted_col].to_numpy(dtype=float)

        if len(y) > 0:
            residual = y - pred
            ss_res = float(np.sum(residual ** 2))
            ss_tot = float(np.sum((y - np.mean(y)) ** 2))
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            rmse = float(np.sqrt(np.mean(residual ** 2)))
            mae = float(np.mean(np.abs(residual)))
            bias = float(np.mean(residual))
            max_abs_residual = float(np.max(np.abs(residual)))
        else:
            residual = np.array([], dtype=float)
            ss_res = ss_tot = r2 = rmse = mae = bias = max_abs_residual = np.nan

        weighted_chi2 = np.nan
        weighted_rmse_scaled = np.nan
        mean_abs_scaled_residual = np.nan
        max_abs_scaled_residual = np.nan

        if len(y) > 0 and sigma_col in sub.columns:
            sigma = sub.loc[included, sigma_col].to_numpy(dtype=float)
            sigma_ok = np.isfinite(sigma) & (sigma > 0)
            if np.any(sigma_ok):
                scaled = residual[sigma_ok] / sigma[sigma_ok]
                weighted_chi2 = float(np.sum(scaled ** 2))
                weighted_rmse_scaled = float(np.sqrt(np.mean(scaled ** 2)))
                mean_abs_scaled_residual = float(np.mean(np.abs(scaled)))
                max_abs_scaled_residual = float(np.max(np.abs(scaled)))

        total_valid_points = int(valid.sum())
        n_fit_points = int(included.sum())

        # Final-point diagnostic, using the latest available time for this condition.
        final_observed = np.nan
        final_predicted = np.nan
        final_residual = np.nan
        final_time = np.nan
        final_rows = sub.loc[valid].sort_values(time_col) if time_col in sub.columns else sub.loc[valid]
        if not final_rows.empty:
            last = final_rows.iloc[-1]
            final_observed = float(last[observed_col])
            final_predicted = float(last[predicted_col])
            final_residual = final_observed - final_predicted
            final_time = float(last[time_col]) if time_col in sub.columns else np.nan

        rows.append({
            "model": model_name,
            "condition": label,
            "n_total_valid_points": total_valid_points,
            "n_fit_points": n_fit_points,
            "r2_fit_points": r2,
            "rmse_fit_points": rmse,
            "mae_fit_points": mae,
            "mean_residual_bias_fit_points": bias,
            "max_abs_residual_fit_points": max_abs_residual,
            "ss_res_fit_points": ss_res,
            "ss_tot_fit_points": ss_tot,
            "weighted_chi2_fit_points": weighted_chi2,
            "weighted_rmse_scaled_residual": weighted_rmse_scaled,
            "mean_abs_scaled_residual": mean_abs_scaled_residual,
            "max_abs_scaled_residual": max_abs_scaled_residual,
            "final_time_min": final_time,
            "final_observed_y": final_observed,
            "final_predicted_y": final_predicted,
            "final_residual_y": final_residual,
        })

    # Global quality across all included points.
    _one_row(fit_points, "__global__")

    # Per-condition quality.
    for cond, sub in fit_points.groupby(condition_col, sort=False):
        _one_row(sub, cond)

    return pd.DataFrame(rows)



def save_tables(long, fit_all, metrics, direct_ratio, output_dir):
    output_dir.mkdir(exist_ok=True, parents=True)

    long.to_csv(output_dir / "parsed_long_normalized_data.csv", index=False)
    fit_all.to_csv(output_dir / "joint_model_fit_points.csv", index=False)
    metrics.to_csv(output_dir / "joint_model_fit_metrics.csv", index=False)
    direct_ratio.to_csv(output_dir / "direct_ratio_corrected_values.csv", index=False)

    fit_quality = compute_fit_quality_summary(
        fit_all,
        model_name="independent_plateau_joint_model",
    )
    fit_quality.to_csv(output_dir / "joint_model_fit_quality.csv", index=False)
    fit_quality.round(4).to_csv(
        output_dir / "joint_model_fit_quality_report_rounded.csv",
        index=False,
    )

    report_cols = [
        "condition",
        "fit_status",
        "k_min_inv",
        "k_se_min_inv",
        "k_ci95_low_min_inv",
        "k_ci95_high_min_inv",
        "t_half_min",
        "t_half_se_min",
        "t_half_ci95_low_min",
        "t_half_ci95_high_min",
        "plateau_P",
        "P_se",
        "P_ci95_low",
        "P_ci95_high",
        "clearable_fraction_1_minus_P",
        "t90_min",
        "t95_min",
        "r2_raw_normalized",
        "observed_y_final",
        "observed_effect_corrected_final",
    ]

    metrics[[c for c in report_cols if c in metrics.columns]].round(4).to_csv(
        output_dir / "joint_model_fit_metrics_report_rounded.csv",
        index=False,
    )

    # Excel export is optional. It requires the optional package openpyxl.
    # If openpyxl is not installed, all CSV outputs above are still written normally.
    try:
        import openpyxl  # noqa: F401

        with pd.ExcelWriter(output_dir / "joint_model_kinetics_analysis.xlsx", engine="openpyxl") as writer:
            metrics.to_excel(writer, sheet_name="metrics", index=False)
            metrics[[c for c in report_cols if c in metrics.columns]].round(4).to_excel(
                writer, sheet_name="metrics_rounded", index=False
            )
            fit_quality.to_excel(writer, sheet_name="fit_quality", index=False)
            fit_all.to_excel(writer, sheet_name="joint_fit_points", index=False)
            direct_ratio.to_excel(writer, sheet_name="direct_ratio_correction", index=False)
            long.to_excel(writer, sheet_name="parsed_long_data", index=False)
    except ModuleNotFoundError:
        print(
            "openpyxl is not installed, so Excel output was skipped. "
            "CSV outputs were saved. To enable Excel output, run: pip install openpyxl"
        )


def make_plots(long, fit_all, direct_ratio, res, unpack, control_label, antibody_labels, output_dir):
    output_dir.mkdir(exist_ok=True, parents=True)

    k_base, p_base, effects = unpack(res.x)

    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    labels = [control_label] + list(antibody_labels)
    color_map = {label: colors[i % len(colors)] for i, label in enumerate(labels)}

    t_max = float(long["fit_time_min"].max())
    t_grid = np.linspace(0, t_max, 500)

    # 1) Raw normalized data and joint model fit
    fig, ax = plt.subplots(figsize=(10, 7))

    mock = long[long["condition"] == control_label].sort_values("fit_time_min")
    ax.plot(
        mock["fit_time_min"],
        mock["mean_normalized_volume"],
        "o",
        color=color_map[control_label],
        markersize=4,
        label=f"{control_label} data",
    )
    ax.plot(
        t_grid,
        baseline_model(t_grid, k_base, p_base),
        "-",
        color=color_map[control_label],
        linewidth=2,
        label=f"{control_label} baseline fit",
    )

    for ab in antibody_labels:
        sub = long[long["condition"] == ab].sort_values("fit_time_min")
        color = color_map[ab]
        k, p = effects[ab]

        ax.plot(
            sub["fit_time_min"],
            sub["mean_normalized_volume"],
            "o",
            color=color,
            markersize=4,
            label=f"{ab} data",
        )
        ax.plot(
            t_grid,
            baseline_model(t_grid, k_base, p_base) * effect_model(t_grid, k, p),
            "-",
            color=color,
            linewidth=2,
            label=f"{ab} joint fit",
        )

    ax.set_xlabel("Elapsed actual time from first measured bin (min)")
    ax.set_ylabel("Normalized interaction-site volume")
    ax.set_title("Global joint model: raw normalized data and fitted curves")
    ax.legend(frameon=False, ncol=2, fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / "joint_model_raw_normalized_data_and_fits.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # 2) Antibody-specific baseline-corrected effect from joint model
    fig, ax = plt.subplots(figsize=(10, 7))

    for ab in antibody_labels:
        sub = fit_all[fit_all["condition"] == ab].sort_values("fit_time_min")
        color = color_map[ab]
        k, p = effects[ab]

        ax.plot(
            sub["fit_time_min"],
            sub["effect_corrected_by_baseline_fit"],
            "o",
            color=color,
            markersize=4,
            label=ab,
        )
        ax.plot(
            t_grid,
            effect_model(t_grid, k, p),
            "-",
            color=color,
            linewidth=2,
        )

    ax.axhline(1, linestyle="--", linewidth=1, color="black", alpha=0.5)
    ax.set_xlabel("Elapsed actual time from first measured bin (min)")
    ax.set_ylabel("Antibody-specific effect, C_norm / fitted baseline")
    ax.set_title("Baseline-corrected antibody effects from joint model")
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    fig.savefig(output_dir / "joint_model_baseline_corrected_antibody_effects.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # 3) Direct measured-mock ratio correction
    fig, ax = plt.subplots(figsize=(10, 7))

    for ab in antibody_labels:
        sub = direct_ratio[direct_ratio["antibody_condition"] == ab].sort_values("fit_time_min")
        color = color_map[ab]
        ax.plot(
            sub["fit_time_min"],
            sub["direct_ratio_corrected_mean"],
            "o",
            color=color,
            markersize=4,
            label=ab,
        )

    ax.axhline(1, linestyle="--", linewidth=1, color="black", alpha=0.5)
    ax.set_xlabel("Elapsed actual time from first measured bin (min)")
    ax.set_ylabel("Direct ratio correction, C_norm / interpolated mock")
    ax.set_title("Direct ratio-corrected antibody curves using measured/interpolated mock")
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    fig.savefig(output_dir / "direct_ratio_corrected_antibody_curves.png", dpi=300, bbox_inches="tight")
    plt.close(fig)


save_tables(long, fit_all, metrics, direct_ratio, OUTPUT_DIR)
make_plots(long, fit_all, direct_ratio, res, unpack, CONTROL_LABEL, ANTIBODY_LABELS, OUTPUT_DIR)

print(f"Saved outputs to: {OUTPUT_DIR.resolve()}")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(path.name)



## Additional model comparison: shared and Durva/Enva-fixed plateau

This section adds two **separate** model outputs.

1. **Strict shared-plateau joint model**  
   Fits a joint model where the listed antibodies share one plateau \(P_\mathrm{shared}\), but each antibody has its own \(k\):

   \[
   C_j(t)=B(t)\left[P_\mathrm{shared}+(1-P_\mathrm{shared})e^{-k_jt}\right]
   \]

2. **Durva/Enva-informed fixed-plateau sensitivity model**  
   First estimates a shared low plateau using only the reference conditions, by default Durva and Enva.  
   Then refits each selected condition with that plateau fixed:

   \[
   C_j(t)=B(t)\left[P_\mathrm{Durva/Enva}+(1-P_\mathrm{Durva/Enva})e^{-k_jt}\right]
   \]

This lets you compare the original independent-plateau model against the hypothesis that weak/incomplete-looking conditions are slower approaches to the same final plateau.


In [ ]:

# -----------------------------
# Shared-plateau and Durva/Enva-fixed plateau model comparison
# -----------------------------

def fit_shared_plateau_joint_model(
    long,
    control_label,
    antibody_labels,
    error_for_fit="sd",
    exclude_t0=True,
    max_bin_dev=None,
):
    """
    Joint model with one shared antibody-effect plateau for all listed antibodies:

        mock:
            y = B(t)

        antibody j:
            y = B(t) * [P_shared + (1 - P_shared) * exp(-k_j * t)]

    Baseline B(t) is fitted together with the antibody curves.
    """
    antibody_labels = list(antibody_labels)
    labels = [control_label] + antibody_labels

    fit_df = make_fit_table(
        long,
        labels,
        error_for_fit=error_for_fit,
        exclude_t0=exclude_t0,
        max_bin_dev=max_bin_dev,
    )
    included = fit_df[fit_df["included_in_fit"]].copy()

    if included.empty:
        raise ValueError("No valid data points are included in the shared-plateau fit.")

    # Baseline initial guesses
    mock_all = long[long["condition"] == control_label].sort_values("fit_time_min")
    p_base0 = float(np.nanmedian(mock_all["mean_normalized_volume"].tail(5)))
    p_base0 = float(np.clip(p_base0, 0.0, min(P_BASE_MAX, 1.2)))
    k_base0 = 0.005

    # Shared plateau initial guess from the final baseline-corrected values of listed antibodies
    # using the baseline initial guess only.
    p_guesses = []
    for ab in antibody_labels:
        ab_all = long[long["condition"] == ab].sort_values("fit_time_min")
        if not ab_all.empty:
            t_tail = ab_all["fit_time_min"].tail(5).to_numpy(dtype=float)
            y_tail = ab_all["mean_normalized_volume"].tail(5).to_numpy(dtype=float)
            b_tail = baseline_model(t_tail, k_base0, p_base0)
            with np.errstate(divide="ignore", invalid="ignore"):
                p_guesses.extend(list(y_tail / b_tail))
    p_shared0 = float(np.nanmedian(p_guesses)) if len(p_guesses) else 0.5
    p_shared0 = float(np.clip(p_shared0, 0.0, P_EFFECT_MAX))

    x0 = [k_base0, p_base0, p_shared0]
    lb = [0.0, 0.0, 0.0]
    ub = [K_BASE_MAX, P_BASE_MAX, P_EFFECT_MAX]

    for _ab in antibody_labels:
        x0 += [0.03]
        lb += [0.0]
        ub += [K_EFFECT_MAX]

    def unpack_shared(params):
        k_base, p_base, p_shared = params[0], params[1], params[2]
        effects = {}
        idx = 3
        for ab in antibody_labels:
            effects[ab] = (params[idx], p_shared)
            idx += 1
        return k_base, p_base, p_shared, effects

    def predict_rows(params, rows):
        k_base, p_base, p_shared, effects = unpack_shared(params)
        t = rows["fit_time_min"].to_numpy(dtype=float)
        b = baseline_model(t, k_base, p_base)
        pred = np.empty(len(rows), dtype=float)
        conds = rows["condition"].to_numpy()

        for i, cond in enumerate(conds):
            if cond == control_label:
                pred[i] = b[i]
            else:
                k, _p = effects[cond]
                pred[i] = b[i] * effect_model(t[i], k, p_shared)

        return pred

    def residuals(params):
        pred = predict_rows(params, included)
        y = included["mean_normalized_volume"].to_numpy(dtype=float)
        sigma = included["sigma"].to_numpy(dtype=float)
        return (pred - y) / sigma

    res_shared = least_squares(
        residuals,
        x0,
        bounds=(lb, ub),
        max_nfev=100000,
        loss="linear",
    )

    m = len(res_shared.fun)
    n_params = len(res_shared.x)
    dof = max(1, m - n_params)
    red_chi2 = float(np.sum(res_shared.fun ** 2) / dof)

    try:
        jtj_inv = np.linalg.pinv(res_shared.jac.T @ res_shared.jac)
        cov_shared = jtj_inv * red_chi2
        se_shared = np.sqrt(np.diag(cov_shared))
    except Exception:
        cov_shared = None
        se_shared = np.full_like(res_shared.x, np.nan, dtype=float)

    fit_all_shared = fit_df.copy()
    fit_all_shared["model_y"] = predict_rows(res_shared.x, fit_all_shared)

    k_base, p_base, p_shared, effects = unpack_shared(res_shared.x)
    fit_all_shared["baseline_y"] = baseline_model(fit_all_shared["fit_time_min"], k_base, p_base)
    fit_all_shared["effect_corrected_by_baseline_fit"] = (
        fit_all_shared["mean_normalized_volume"] / fit_all_shared["baseline_y"]
    )
    fit_all_shared.loc[
        fit_all_shared["condition"] == control_label,
        "effect_corrected_by_baseline_fit"
    ] = np.nan
    fit_all_shared["plateau_model"] = "strict_shared_plateau"

    return res_shared, cov_shared, se_shared, fit_all_shared, unpack_shared, red_chi2


def compute_shared_plateau_metrics(
    long,
    fit_all_shared,
    res_shared,
    se_shared,
    unpack_shared,
    control_label,
    antibody_labels,
    model_name="strict_shared_plateau",
):
    """
    Metrics for the strict shared-plateau model.
    """
    antibody_labels = list(antibody_labels)

    k_base, p_base, p_shared, effects = unpack_shared(res_shared.x)

    param_names = ["k_base", "P_base", "P_shared"] + [f"{ab}_k" for ab in antibody_labels]
    se_map = dict(zip(param_names, se_shared))

    rows = []

    mock_obs = long[long["condition"] == control_label].sort_values("fit_time_min")
    base_row = {
        "model": model_name,
        "condition": control_label,
        "role": "baseline_control",
        "fit_status": "baseline fitted",
        "shared_plateau_P": np.nan,
        "clearable_fraction_1_minus_P": np.nan,
        "t90_min": np.nan,
        "t95_min": np.nan,
        "observed_y_final": float(mock_obs["mean_normalized_volume"].iloc[-1]),
        "observed_min_y": float(mock_obs["mean_normalized_volume"].min()),
        "observed_effect_corrected_final": np.nan,
        "observed_effect_corrected_min": np.nan,
        "n_fit_points": int(fit_all_shared[(fit_all_shared["condition"] == control_label) & fit_all_shared["included_in_fit"]].shape[0]),
        "r2_raw_normalized": np.nan,
        "rmse_raw_normalized": np.nan,
    }

    base_row = add_rate_uncertainty_fields(
        base_row,
        k=k_base,
        k_se=se_map.get("k_base", np.nan),
        p=p_base,
        p_se=se_map.get("P_base", np.nan),
        has_interpretable_half_time=True,
    )
    rows.append(base_row)

    p_shared_se = se_map.get("P_shared", np.nan)
    p_shared_ci_low, p_shared_ci_high = symmetric_ci_from_se(
        p_shared,
        p_shared_se,
        lower_bound=0.0,
        upper_bound=P_EFFECT_MAX,
    )

    for ab in antibody_labels:
        k, _p = effects[ab]
        k_se = se_map.get(f"{ab}_k", np.nan)

        has_decay = (np.isfinite(k) and k > MIN_INTERPRETABLE_K) and (p_shared < 1)
        t90 = np.log(10) / k if has_decay else np.nan
        t95 = np.log(20) / k if has_decay else np.nan

        sub = fit_all_shared[fit_all_shared["condition"] == ab].sort_values("fit_time_min").copy()
        y = sub["mean_normalized_volume"].to_numpy(dtype=float)
        pred = sub["model_y"].to_numpy(dtype=float)

        mask = sub["fit_time_min"].to_numpy(dtype=float) > 0
        if mask.sum() > 1:
            ss_res = np.sum((y[mask] - pred[mask]) ** 2)
            ss_tot = np.sum((y[mask] - np.mean(y[mask])) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            rmse = np.sqrt(np.mean((y[mask] - pred[mask]) ** 2))
        else:
            r2 = np.nan
            rmse = np.nan

        fit_status = "shared_plateau_decay_fit_ok" if has_decay else "no_decay_or_poor_shared_plateau_fit"

        row = {
            "model": model_name,
            "condition": ab,
            "role": "antibody",
            "fit_status": fit_status,
            "shared_plateau_P": p_shared,
            "shared_plateau_P_se": p_shared_se,
            "shared_plateau_P_ci95_low": p_shared_ci_low,
            "shared_plateau_P_ci95_high": p_shared_ci_high,
            "clearable_fraction_1_minus_P": 1 - p_shared if has_decay else np.nan,
            "t90_min": t90,
            "t95_min": t95,
            "observed_y_final": float(sub["mean_normalized_volume"].iloc[-1]),
            "observed_min_y": float(sub["mean_normalized_volume"].min()),
            "observed_effect_corrected_final": float(sub["effect_corrected_by_baseline_fit"].iloc[-1]),
            "observed_effect_corrected_min": float(sub["effect_corrected_by_baseline_fit"].min()),
            "n_fit_points": int(sub[sub["included_in_fit"]].shape[0]),
            "r2_raw_normalized": r2,
            "rmse_raw_normalized": rmse,
        }

        row = add_rate_uncertainty_fields(
            row,
            k=k,
            k_se=k_se,
            p=p_shared,
            p_se=p_shared_se,
            has_interpretable_half_time=has_decay,
        )

        rows.append(row)

    return pd.DataFrame(rows)


def fit_fixed_plateau_sensitivity(
    long,
    control_label,
    antibody_labels,
    k_base,
    p_base,
    p_fixed,
    p_fixed_se=np.nan,
    p_fixed_ci_low=np.nan,
    p_fixed_ci_high=np.nan,
    error_for_fit="sd",
    exclude_t0=True,
    max_bin_dev=None,
):
    """
    Fit each antibody with the baseline fixed and the antibody-effect plateau fixed.

    Only k_j is fitted for each condition:
        C_j(t) = B_fixed(t) * [P_fixed + (1 - P_fixed) * exp(-k_j * t)]

    This is a sensitivity analysis, not a replacement for the independent-plateau model.
    """
    rows_metrics = []
    fit_rows = []

    for ab in antibody_labels:
        fit_df = make_fit_table(
            long,
            [ab],
            error_for_fit=error_for_fit,
            exclude_t0=exclude_t0,
            max_bin_dev=max_bin_dev,
        )
        included = fit_df[fit_df["included_in_fit"]].copy()

        if included.empty:
            continue

        t = included["fit_time_min"].to_numpy(dtype=float)
        y = included["mean_normalized_volume"].to_numpy(dtype=float)
        sigma = included["sigma"].to_numpy(dtype=float)

        def predict_k(k, rows):
            tt = rows["fit_time_min"].to_numpy(dtype=float)
            return baseline_model(tt, k_base, p_base) * effect_model(tt, k, p_fixed)

        def residuals_k(params):
            return (predict_k(params[0], included) - y) / sigma

        res_k = least_squares(
            residuals_k,
            x0=[0.03],
            bounds=([0.0], [K_EFFECT_MAX]),
            max_nfev=100000,
            loss="linear",
        )

        k = float(res_k.x[0])
        m = len(res_k.fun)
        n_params = 1
        dof = max(1, m - n_params)
        red_chi2 = float(np.sum(res_k.fun ** 2) / dof)

        try:
            jtj_inv = np.linalg.pinv(res_k.jac.T @ res_k.jac)
            cov_k = jtj_inv * red_chi2
            k_se = float(np.sqrt(np.diag(cov_k))[0])
        except Exception:
            k_se = np.nan

        sub_all = fit_df.copy()
        sub_all["model_y"] = predict_k(k, sub_all)
        sub_all["baseline_y"] = baseline_model(sub_all["fit_time_min"], k_base, p_base)
        sub_all["effect_corrected_by_baseline_fit"] = (
            sub_all["mean_normalized_volume"] / sub_all["baseline_y"]
        )
        sub_all["fixed_plateau_P"] = p_fixed
        sub_all["plateau_model"] = "durva_enva_fixed_plateau_sensitivity"
        fit_rows.append(sub_all)

        y_all = sub_all["mean_normalized_volume"].to_numpy(dtype=float)
        pred_all = sub_all["model_y"].to_numpy(dtype=float)
        mask = sub_all["fit_time_min"].to_numpy(dtype=float) > 0
        if mask.sum() > 1:
            ss_res = np.sum((y_all[mask] - pred_all[mask]) ** 2)
            ss_tot = np.sum((y_all[mask] - np.mean(y_all[mask])) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
            rmse = np.sqrt(np.mean((y_all[mask] - pred_all[mask]) ** 2))
        else:
            r2 = np.nan
            rmse = np.nan

        has_decay = (np.isfinite(k) and k > MIN_INTERPRETABLE_K) and (p_fixed < 1)
        t90 = np.log(10) / k if has_decay else np.nan
        t95 = np.log(20) / k if has_decay else np.nan

        row = {
            "model": "durva_enva_fixed_plateau_sensitivity",
            "condition": ab,
            "role": "antibody",
            "fit_status": "fixed_plateau_decay_fit_ok" if has_decay else "no_decay_or_poor_fixed_plateau_fit",
            "fixed_plateau_P": p_fixed,
            "fixed_plateau_P_se_from_reference": p_fixed_se,
            "fixed_plateau_P_ci95_low_from_reference": p_fixed_ci_low,
            "fixed_plateau_P_ci95_high_from_reference": p_fixed_ci_high,
            "clearable_fraction_1_minus_P": 1 - p_fixed if has_decay else np.nan,
            "t90_min": t90,
            "t95_min": t95,
            "observed_y_final": float(sub_all["mean_normalized_volume"].iloc[-1]),
            "observed_min_y": float(sub_all["mean_normalized_volume"].min()),
            "observed_effect_corrected_final": float(sub_all["effect_corrected_by_baseline_fit"].iloc[-1]),
            "observed_effect_corrected_min": float(sub_all["effect_corrected_by_baseline_fit"].min()),
            "n_fit_points": int(sub_all[sub_all["included_in_fit"]].shape[0]),
            "r2_raw_normalized": r2,
            "rmse_raw_normalized": rmse,
            "red_chi2": red_chi2,
        }

        row = add_rate_uncertainty_fields(
            row,
            k=k,
            k_se=k_se,
            p=p_fixed,
            p_se=p_fixed_se,
            has_interpretable_half_time=has_decay,
        )
        rows_metrics.append(row)

    metrics_fixed = pd.DataFrame(rows_metrics)
    fit_fixed = pd.concat(fit_rows, ignore_index=True) if fit_rows else pd.DataFrame()
    return metrics_fixed, fit_fixed


def save_shared_plateau_outputs(
    shared_metrics,
    shared_fit_all,
    fixed_metrics,
    fixed_fit_all,
    output_dir,
):
    output_dir.mkdir(exist_ok=True, parents=True)

    shared_metrics.to_csv(output_dir / "shared_plateau_joint_model_fit_metrics.csv", index=False)
    shared_fit_all.to_csv(output_dir / "shared_plateau_joint_model_fit_points.csv", index=False)
    fixed_metrics.to_csv(output_dir / "durva_enva_fixed_plateau_sensitivity_metrics.csv", index=False)
    fixed_fit_all.to_csv(output_dir / "durva_enva_fixed_plateau_sensitivity_fit_points.csv", index=False)

    shared_fit_quality = compute_fit_quality_summary(
        shared_fit_all,
        model_name="strict_shared_plateau_joint_model",
    )
    shared_fit_quality.to_csv(
        output_dir / "shared_plateau_joint_model_fit_quality.csv",
        index=False,
    )
    shared_fit_quality.round(4).to_csv(
        output_dir / "shared_plateau_joint_model_fit_quality_report_rounded.csv",
        index=False,
    )

    fixed_fit_quality = compute_fit_quality_summary(
        fixed_fit_all,
        model_name="durva_enva_fixed_plateau_sensitivity",
    )
    fixed_fit_quality.to_csv(
        output_dir / "durva_enva_fixed_plateau_sensitivity_fit_quality.csv",
        index=False,
    )
    fixed_fit_quality.round(4).to_csv(
        output_dir / "durva_enva_fixed_plateau_sensitivity_fit_quality_report_rounded.csv",
        index=False,
    )

    report_cols = [
        "model",
        "condition",
        "fit_status",
        "k_min_inv",
        "k_se_min_inv",
        "k_ci95_low_min_inv",
        "k_ci95_high_min_inv",
        "t_half_min",
        "t_half_se_min",
        "t_half_ci95_low_min",
        "t_half_ci95_high_min",
        "plateau_P",
        "P_se",
        "P_ci95_low",
        "P_ci95_high",
        "shared_plateau_P",
        "shared_plateau_P_se",
        "shared_plateau_P_ci95_low",
        "shared_plateau_P_ci95_high",
        "fixed_plateau_P",
        "fixed_plateau_P_se_from_reference",
        "fixed_plateau_P_ci95_low_from_reference",
        "fixed_plateau_P_ci95_high_from_reference",
        "clearable_fraction_1_minus_P",
        "t90_min",
        "t95_min",
        "r2_raw_normalized",
        "observed_effect_corrected_final",
    ]

    shared_metrics[[c for c in report_cols if c in shared_metrics.columns]].round(4).to_csv(
        output_dir / "shared_plateau_joint_model_fit_metrics_report_rounded.csv",
        index=False,
    )
    fixed_metrics[[c for c in report_cols if c in fixed_metrics.columns]].round(4).to_csv(
        output_dir / "durva_enva_fixed_plateau_sensitivity_metrics_report_rounded.csv",
        index=False,
    )

    # Optional Excel add-on if openpyxl is available.
    try:
        import openpyxl  # noqa: F401

        with pd.ExcelWriter(output_dir / "shared_and_fixed_plateau_model_outputs.xlsx", engine="openpyxl") as writer:
            shared_metrics.to_excel(writer, sheet_name="shared_plateau_metrics", index=False)
            fixed_metrics.to_excel(writer, sheet_name="fixed_plateau_metrics", index=False)
            shared_fit_quality.to_excel(writer, sheet_name="shared_fit_quality", index=False)
            fixed_fit_quality.to_excel(writer, sheet_name="fixed_fit_quality", index=False)
            shared_fit_all.to_excel(writer, sheet_name="shared_fit_points", index=False)
            fixed_fit_all.to_excel(writer, sheet_name="fixed_fit_points", index=False)
    except ModuleNotFoundError:
        pass


def make_shared_plateau_plots(
    long,
    shared_fit_all,
    shared_res,
    unpack_shared,
    shared_labels,
    fixed_fit_all,
    fixed_metrics,
    control_label,
    output_dir,
):
    output_dir.mkdir(exist_ok=True, parents=True)

    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    all_labels = [control_label] + list(ANTIBODY_LABELS)
    color_map = {label: colors[i % len(colors)] for i, label in enumerate(all_labels)}

    t_max = float(long["fit_time_min"].max())
    t_grid = np.linspace(0, t_max, 500)

    # Strict shared-plateau raw normalized fit plot
    k_base_s, p_base_s, p_shared, effects_s = unpack_shared(shared_res.x)

    fig, ax = plt.subplots(figsize=(10, 7))

    mock = long[long["condition"] == control_label].sort_values("fit_time_min")
    ax.plot(
        mock["fit_time_min"],
        mock["mean_normalized_volume"],
        "o",
        color=color_map[control_label],
        markersize=4,
        label=f"{control_label} data",
    )
    ax.plot(
        t_grid,
        baseline_model(t_grid, k_base_s, p_base_s),
        "-",
        color=color_map[control_label],
        linewidth=2,
        label=f"{control_label} baseline fit",
    )

    for ab in shared_labels:
        sub = long[long["condition"] == ab].sort_values("fit_time_min")
        color = color_map.get(ab, None)
        k, _p = effects_s[ab]

        ax.plot(sub["fit_time_min"], sub["mean_normalized_volume"], "o", color=color, markersize=4, label=f"{ab} data")
        ax.plot(
            t_grid,
            baseline_model(t_grid, k_base_s, p_base_s) * effect_model(t_grid, k, p_shared),
            "-",
            color=color,
            linewidth=2,
            label=f"{ab} shared-P fit",
        )

    ax.set_xlabel("Elapsed actual time from first measured bin (min)")
    ax.set_ylabel("Normalized interaction-site volume")
    ax.set_title("Strict shared-plateau joint model")
    ax.legend(frameon=False, ncol=2, fontsize=8)
    fig.tight_layout()
    fig.savefig(output_dir / "shared_plateau_joint_model_raw_normalized_data_and_fits.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Strict shared-plateau baseline-corrected effects
    fig, ax = plt.subplots(figsize=(10, 7))
    for ab in shared_labels:
        sub = shared_fit_all[shared_fit_all["condition"] == ab].sort_values("fit_time_min")
        color = color_map.get(ab, None)
        k, _p = effects_s[ab]

        ax.plot(
            sub["fit_time_min"],
            sub["effect_corrected_by_baseline_fit"],
            "o",
            color=color,
            markersize=4,
            label=ab,
        )
        ax.plot(t_grid, effect_model(t_grid, k, p_shared), "-", color=color, linewidth=2)

    ax.axhline(1, linestyle="--", linewidth=1, color="black", alpha=0.5)
    ax.axhline(p_shared, linestyle=":", linewidth=1.5, color="black", alpha=0.7, label=f"shared P={p_shared:.3f}")
    ax.set_xlabel("Elapsed actual time from first measured bin (min)")
    ax.set_ylabel("Antibody-specific effect, C_norm / fitted baseline")
    ax.set_title("Strict shared-plateau antibody effects")
    ax.legend(frameon=False, ncol=2)
    fig.tight_layout()
    fig.savefig(output_dir / "shared_plateau_joint_model_baseline_corrected_effects.png", dpi=300, bbox_inches="tight")
    plt.close(fig)

    # Durva/Enva-fixed plateau sensitivity plot
    if not fixed_fit_all.empty:
        fig, ax = plt.subplots(figsize=(10, 7))

        p_fixed = float(fixed_metrics["fixed_plateau_P"].dropna().iloc[0]) if "fixed_plateau_P" in fixed_metrics.columns and fixed_metrics["fixed_plateau_P"].notna().any() else np.nan

        for ab in fixed_fit_all["condition"].dropna().unique():
            sub = fixed_fit_all[fixed_fit_all["condition"] == ab].sort_values("fit_time_min")
            color = color_map.get(ab, None)
            met = fixed_metrics[fixed_metrics["condition"] == ab]
            if met.empty or not np.isfinite(met["k_min_inv"].iloc[0]):
                continue
            k = float(met["k_min_inv"].iloc[0])

            ax.plot(
                sub["fit_time_min"],
                sub["effect_corrected_by_baseline_fit"],
                "o",
                color=color,
                markersize=4,
                label=ab,
            )
            ax.plot(t_grid, effect_model(t_grid, k, p_fixed), "-", color=color, linewidth=2)

        ax.axhline(1, linestyle="--", linewidth=1, color="black", alpha=0.5)
        if np.isfinite(p_fixed):
            ax.axhline(p_fixed, linestyle=":", linewidth=1.5, color="black", alpha=0.7, label=f"fixed P={p_fixed:.3f}")

        ax.set_xlabel("Elapsed actual time from first measured bin (min)")
        ax.set_ylabel("Antibody-specific effect, C_norm / independent-model baseline")
        ax.set_title("Durva/Enva-fixed plateau sensitivity model")
        ax.legend(frameon=False, ncol=2)
        fig.tight_layout()
        fig.savefig(output_dir / "durva_enva_fixed_plateau_sensitivity_baseline_corrected_effects.png", dpi=300, bbox_inches="tight")
        plt.close(fig)


# ---- Run additional models ----

# 1) Strict shared-plateau model across selected antibody conditions.
shared_res, shared_cov, shared_se, shared_fit_all, unpack_shared, shared_red_chi2 = fit_shared_plateau_joint_model(
    long,
    control_label=CONTROL_LABEL,
    antibody_labels=SHARED_PLATEAU_MODEL_LABELS,
    error_for_fit=ERROR_FOR_FIT,
    exclude_t0=EXCLUDE_T0_FROM_FIT,
    max_bin_dev=MAX_ALLOWED_BIN_DEVIATION_MIN,
)

shared_metrics = compute_shared_plateau_metrics(
    long=long,
    fit_all_shared=shared_fit_all,
    res_shared=shared_res,
    se_shared=shared_se,
    unpack_shared=unpack_shared,
    control_label=CONTROL_LABEL,
    antibody_labels=SHARED_PLATEAU_MODEL_LABELS,
    model_name="strict_shared_plateau",
)

# 2) Reference-only shared plateau from Durva/Enva.
reference_res, reference_cov, reference_se, reference_fit_all, unpack_reference, reference_red_chi2 = fit_shared_plateau_joint_model(
    long,
    control_label=CONTROL_LABEL,
    antibody_labels=SHARED_PLATEAU_REFERENCE_LABELS,
    error_for_fit=ERROR_FOR_FIT,
    exclude_t0=EXCLUDE_T0_FROM_FIT,
    max_bin_dev=MAX_ALLOWED_BIN_DEVIATION_MIN,
)

_reference_k_base, _reference_p_base, p_fixed_from_reference, _reference_effects = unpack_reference(reference_res.x)
reference_param_names = ["k_base", "P_base", "P_shared"] + [f"{ab}_k" for ab in SHARED_PLATEAU_REFERENCE_LABELS]
reference_se_map = dict(zip(reference_param_names, reference_se))
p_fixed_se = reference_se_map.get("P_shared", np.nan)
p_fixed_ci_low, p_fixed_ci_high = symmetric_ci_from_se(
    p_fixed_from_reference,
    p_fixed_se,
    lower_bound=0.0,
    upper_bound=P_EFFECT_MAX,
)

# Use the baseline from the original independent-plateau joint model for fixed-plateau sensitivity.
# This keeps the sensitivity test comparable to the original primary output.
primary_k_base, primary_p_base, _primary_effects = unpack(res.x)

fixed_metrics, fixed_fit_all = fit_fixed_plateau_sensitivity(
    long=long,
    control_label=CONTROL_LABEL,
    antibody_labels=FIXED_PLATEAU_APPLY_LABELS,
    k_base=primary_k_base,
    p_base=primary_p_base,
    p_fixed=p_fixed_from_reference,
    p_fixed_se=p_fixed_se,
    p_fixed_ci_low=p_fixed_ci_low,
    p_fixed_ci_high=p_fixed_ci_high,
    error_for_fit=ERROR_FOR_FIT,
    exclude_t0=EXCLUDE_T0_FROM_FIT,
    max_bin_dev=MAX_ALLOWED_BIN_DEVIATION_MIN,
)

save_shared_plateau_outputs(
    shared_metrics=shared_metrics,
    shared_fit_all=shared_fit_all,
    fixed_metrics=fixed_metrics,
    fixed_fit_all=fixed_fit_all,
    output_dir=OUTPUT_DIR,
)

make_shared_plateau_plots(
    long=long,
    shared_fit_all=shared_fit_all,
    shared_res=shared_res,
    unpack_shared=unpack_shared,
    shared_labels=SHARED_PLATEAU_MODEL_LABELS,
    fixed_fit_all=fixed_fit_all,
    fixed_metrics=fixed_metrics,
    control_label=CONTROL_LABEL,
    output_dir=OUTPUT_DIR,
)

print("Additional shared/fixed-plateau outputs saved.")
print(f"Strict shared-plateau reduced chi-square-like value: {shared_red_chi2:.4g}")
print(f"Durva/Enva-inferred fixed plateau P = {p_fixed_from_reference:.4f} ± {p_fixed_se:.4f} SE")

display_cols_shared = [
    "model",
    "condition",
    "fit_status",
    "k_min_inv",
    "k_se_min_inv",
    "k_ci95_low_min_inv",
    "k_ci95_high_min_inv",
    "t_half_min",
    "t_half_ci95_low_min",
    "t_half_ci95_high_min",
    "plateau_P",
    "clearable_fraction_1_minus_P",
    "r2_raw_normalized",
    "observed_effect_corrected_final",
]

print("\nStrict shared-plateau model:")
display(shared_metrics[[c for c in display_cols_shared if c in shared_metrics.columns]].round(4))

print("\nDurva/Enva-fixed plateau sensitivity model:")
display(fixed_metrics[[c for c in display_cols_shared if c in fixed_metrics.columns]].round(4))


In [ ]:

# -----------------------------
# Optional: show saved plots in the notebook
# -----------------------------

from IPython.display import Image, display

for plot_name in [
    "joint_model_raw_normalized_data_and_fits.png",
    "joint_model_baseline_corrected_antibody_effects.png",
    "direct_ratio_corrected_antibody_curves.png",
    "shared_plateau_joint_model_raw_normalized_data_and_fits.png",
    "shared_plateau_joint_model_baseline_corrected_effects.png",
    "durva_enva_fixed_plateau_sensitivity_baseline_corrected_effects.png",
]:
    plot_path = OUTPUT_DIR / plot_name
    if plot_path.exists():
        print(plot_name)
        display(Image(filename=str(plot_path)))
